In [9]:
import numpy as np
import pandas as pd
from pid.utils.generate import generate_cov_from_config, random_rotation_mxy, merge_covs
from pid.optimizers import exact_gauss_tilde_pid, exact_gauss_thin_pid

In [10]:
# Starting from 2-D case
dm, dx, dy = 2, 2, 2
theta = 0
gain_y = np.sqrt(2)
gains = np.linspace(0.5, 3, 10)
gains[3] = 0.99  # A gain of 1 is unstable

In [13]:
for i, gain_x in enumerate(gains):
    cov, *dims = generate_cov_from_config(d=(dm, dx, dy), gain=(gain_x, gain_y), theta=theta)
    # Generate random samples from the multivariate Gaussian distribution
    num_samples = 10  # Number of samples to generate
    mean = np.zeros(sum(dims))
    samples = np.random.multivariate_normal(mean, cov, num_samples)
    # print(f'{samples} \n')

In [14]:
# Extend to multi-dimensional case
gains = np.linspace(0, 3, 10)
gains[3] = 0.99  # A gain of 1 is unstable
#gains = np.array([3.0,])
num_doubles = 3
random_rotn = True

# for simplicity, I only print gains[0]
gain = gains[0]
for j in range(num_doubles):
    if j == 0:
        dm, dx, dy = 2, 2, 2
        hx = np.array([[gain, 0], [0, 1]])
        hy = np.array([[1, 0], [0, np.sqrt(2)]])
        sigm = np.eye(dm)
        sigx_m = np.eye(dx)
        sigy_m = np.eye(dy)
        sigw = np.zeros((dx, dy))
        cov = np.block([[sigm, sigm @ hx.T, sigm @ hy.T],
                        [hx @ sigm, hx @ sigm @ hx.T + sigx_m, hx @ sigm @ hy.T + sigw],
                        [hy @ sigm, hy @ sigm @ hx.T + sigw.T, hy @ sigm @ hy.T + sigy_m]])
        cov_old = cov.copy()
    else:
        cov, dm, dx, dy = merge_covs(cov_old, cov_old.copy(), dm, dx, dy)
        cov_old = cov.copy()

    if random_rotn:
        cov = random_rotation_mxy(cov, dm, dx, dy)

    num_samples = 1000  # Number of samples to generate
    mean = np.zeros(sum([dm, dx, dy]))
    samples = np.random.multivariate_normal(mean, cov, num_samples)
    print(f'j: {j}, samples.shape: {samples.shape} \n')

    m = samples[:, :dm]
    x = samples[:, dm:dm+dx]
    y = samples[:, dm+dx:]
    mxy = np.vstack((m.T, x.T, y.T))
    cov = np.corrcoef(mxy)    # Shape: (dm+dx+dy, dm+dx+dy)
    print(f'j: {j}, cov.shape: {cov.shape} \n')

    ret = exact_gauss_tilde_pid(cov, dm, dx, dy)
    print(f'tilde pid: {ret[7]}, {ret[5]}, {ret[6]}, {ret[8]} \n')

    ret = exact_gauss_thin_pid(cov, dm, dx, dy)
    print(f'thin pid: {ret[7]}, {ret[5]}, {ret[6]}, {ret[8]} \n')

    dm = dx = dy = 2**(j+1)

j: 0, samples.shape: (1000, 6) 

j: 0, cov.shape: (6, 6) 

tilde pid: 0.4439371624542434, 0.0, 0.8638885954420541, 0.17445079793382456 

thin pid: 0.4439371624448709, 9.372502773885572e-12, 0.8638885954514266, 0.17445079792445206 

j: 1, samples.shape: (1000, 12) 

j: 1, cov.shape: (12, 12) 

tilde pid: 1.0142431648432932, 0.0, 1.5186720286787514, 0.4503032024880427 

thin pid: 1.0142431639098146, 9.334786277293006e-10, 1.51867202961223, 0.45030320155456405 

j: 2, samples.shape: (1000, 24) 

j: 2, cov.shape: (24, 24) 

tilde pid: 2.090186143554691, 0.0, 3.16554052722959, 0.8644149980901537 

thin pid: 2.0901861407654128, 2.789278141790419e-09, 3.165540530018868, 0.8644149953008755 



In [15]:
m = samples[:, :dm]
x = samples[:, dm:dm+dx]
y = samples[:, dm+dx:]
mxy = np.vstack((m.T, x.T, y.T))
cov = np.corrcoef(mxy)    # Shape: (dm+dx+dy, dm+dx+dy)
print(cov.shape)

ret = exact_gauss_tilde_pid(cov, dm, dx, dy)
print(ret[7], ret[5], ret[6], ret[8])

ret = exact_gauss_thin_pid(cov, dm, dx, dy)
print(ret[7], ret[5], ret[6], ret[8])

(24, 24)
2.090186143554691 0.0 3.16554052722959 0.8644149980901537
2.0901861407654128 2.789278141790419e-09 3.165540530018868 0.8644149953008755
